# Chapter 04 — Scrum Built the Training Set

**Companion to Applied AI**

Question: What does a verifier buy you when candidates are cheap and noisy?

By the end of this notebook you will have:

- built a miniature spec/verifier/solution dataset
- generated noisy candidates and filtered them deterministically
- measured survivor yield as the mechanism that makes data usable

## What this notebook demonstrates
A synthetic miniature of the chapter's spec→verifier→solution triple. Candidates are deliberately noisy; the verifier — not candidate cleverness — does the work.

In [1]:
SEED = 42
import random
random.seed(SEED)
print("seed:", SEED)

seed: 42


## 1. Specs with checkable verifiers

In [2]:
specs = [
    {"id": "reverse", "desc": "return the reverse of the input string",
     "check": lambda inp, out: out == inp[::-1],
     "inputs": ["hello", "abcd", "12345"]},
    {"id": "normalize", "desc": "lowercase and strip surrounding whitespace",
     "check": lambda inp, out: out == inp.strip().lower(),
     "inputs": ["  Hello ", "MiXeD", "\tPad\n"]},
    {"id": "sumints", "desc": "sum comma-separated integers",
     "check": lambda inp, out: out == str(sum(int(x) for x in inp.split(","))),
     "inputs": ["1,2,3", "10,20", "7,7,7,7"]},
]
print([s["id"] for s in specs])

['reverse', 'normalize', 'sumints']


## 2. Noisy candidate generator (the stand-in for cheap proposals)

In [3]:
def noisy_candidate(spec_id: str, inp: str) -> str:
    r = random.Random(hash((SEED, spec_id, inp)) % (2**32))
    if spec_id == "reverse":
        good = inp[::-1]
    elif spec_id == "normalize":
        good = inp.strip().lower()
    else:
        good = str(sum(int(x) for x in inp.split(",")))
    if r.random() < 0.55:  # correct most of the time, noisy otherwise
        return good
    return good + "?"  # plausible-looking corruption

N_CAND = 30
rows = []
for s in specs:
    for inp in s["inputs"]:
        for k in range(N_CAND):
            out = noisy_candidate(s["id"], inp)
            rows.append((s["id"], inp, out, bool(s["check"](inp, out))))
print(f"generated {len(rows)} candidates")
base_rate = sum(1 for r in rows if r[3]) / len(rows)
print(f"raw candidate accuracy: {base_rate:.2f}")

generated 270 candidates
raw candidate accuracy: 0.78


## 3. The verifier turns volume into a usable survivor set

In [4]:
survivors = [r for r in rows if r[3]]
coverage = {s["id"] for s in specs if any(r[0] == s["id"] and r[3] for r in rows)}
print(f"survivors: {len(survivors)}/{len(rows)}")
print("specs with >=1 verified solution:", sorted(coverage))
assert len(survivors) < len(rows)
assert coverage == {"reverse", "normalize", "sumints"}

survivors: 210/270
specs with >=1 verified solution: ['normalize', 'reverse', 'sumints']


## Interpretation
- Supports: with a checkable verifier, many noisy proposals collapse to a trustworthy set; checkability is the scarce resource.
- Does NOT support: claims about real training data or any specific model.

## Try it yourself
1. Drop the candidate accuracy to 20% and watch survivors shrink but stay correct.
2. Weaken a `check` (e.g. accept near-misses) and count false survivors.
3. Add a fourth spec whose verifier you cannot write — that gap is the chapter's point.